<div style="text-align: center;">
  <span style="
      color: lightskyblue;
      font-family: 'Segoe UI', 'Noto Sans KR', 'Malgun Gothic', sans-serif;
      font-weight: 700;
      font-size: 48px;
      letter-spacing: 0.5px;
      line-height: 1.2;
  ">
    Demo File 
  </span>
</div>

Revised version of Demo_260826

<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Import Library
</span>

In [1]:
import rby1_sdk as rby
import numpy as np
import logging
import time
import pyrealsense2 as rs
import cv2

import sys
from pathlib import Path
# Path.cwd()           : folder "260903", 
# Path.cwd().parents[0]: folder "Demonstration", 
# Path.cwd().parents[1]: folder "RBY1_library_ysw"
sys.path.append(str(Path.cwd().parents[1]))                  
import rby1_lib_ysw as ysw
import functions_260903 as fn

np.set_printoptions(suppress=True, precision=4, linewidth=300)  

# In .ipynb, show logging 
logging.basicConfig(
    level=logging.INFO,
    format="[%(levelname)s] %(message)s",
    # format="%(asctime)s [%(levelname)s] %(message)s",
    stream=sys.stdout,
    force=True,
)

<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Tuning Parameters
</span>

In [2]:
state = 0    # Scenario Sequence               

position_control_minimum_time        = 4      # [sec]

# Vision for Table
vision_record_duration_tableDistance = 5.0    # [sec]
offset_table_distance                = 0.4    # [m]

vision_record_duration_checkBox = 5.0         # [sec]
box_minimum_area = 10000                      # [pixel^2]
box_maximum_distance = 1.0                    # [m]

vision_record_duration_checkObject = 5.0      # [sec]
object_minimum_area = 50                      # [pixel^2]
object_maximum_area = 1500                    # [pixel^2]
object_maximum_distance = 1.0                 # [m]


# Position Control
object_manual_offset_downToGrasp = -0.005      # [m] for z-direction offset when grasp
object_pregrasp_z_offset         = +0.1        # [m]
object_grasp_x_offset            = +0.015      # [m]
box_pregrasp_z_offset            = +0.05       # [m]
box_grasp_x_offset               = -0.04       # [m] outer side center point에서, 조금 더 앞쪽을 잡기 위한 offset 
box_grasp_y_offset               = 0.05        # [m] 그리퍼가 양쪽 모두 바깥 side보다 더 바깥쪽으로 배치되는 경향을 보정
box_grasp_z_offset               = -0.065      # [m]

offset_gripper_open_m = 0.015      # [m]
grasp_current_obj     = 0.15       # [A]
object_floating_z     = 0.117 - object_manual_offset_downToGrasp     # [m]

grasp_current_box             = 0.25      # [A]
box_grasp_gripper_open_pos_mm = 80.0      # [mm]
box_grasp_floating_offset     = 0.08      # [m]

box_move_readyPosition_right = np.array([0.55, -0.25, 1.2])     # desired position of "ee_finger_r1" from "base"  
box_move_readyPosition_left  = np.array([0.55, 0.25, 1.2])      # desired position of "ee_finger_l1" from "base"  


# Mobile Move
moved_distance_to_back  = -0.7    # [m]
desired_yaw_deg_to_turn = -176    # [deg], (+): CCW, (-): CW


# Vision for Shelf
vision_record_duration_shelfDistance = 5.0   # [sec]
offset_shelf_distance = 0.45                 # [m]


# Gripper: current control 
gripper_open_ampere = -0.1    # [A]  
gripper_close_ampere = 0.1    # [A]




# ============================================================
# ROI setting 
# Pixel Origin: (0,0) = 왼쪽 위 꼭짓점
# ============================================================

# table까지 거리를 잴 때 사용하는 ROI (Posture: PACK_POS)
pixel_roi_x_tableDistance = [280, ysw.D435I_PIXEL_WIDTH - 280/2.0] 
pixel_roi_y_tableDistance = [320, ysw.D435I_PIXEL_HEIGHT]    

# table위 물체 detection에 사용하는 ROI (Posture: DEMO_BASIC_POS_260826)
pixel_roi_x_tableTop = None
pixel_roi_y_tableTop = [150, ysw.D435I_PIXEL_HEIGHT - 50]

# # 박스를 grasp하고, 180도 돌아서 shelf까지 갈 때 사용하는 ROI 
pixel_roi_x_shelfDistance = [130, ysw.D435I_PIXEL_WIDTH - 130]      
pixel_roi_y_shelfDistance = [200, ysw.D435I_PIXEL_HEIGHT ]    


<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Initialization [Connect + PowerON]
</span>

In [3]:
# ==========================================================================================================
# [Robot Initialization]
# ==========================================================================================================
robotConfig = ysw.setRobotConfig(ysw.MODE_DIRECT_UPC, ysw.MODEL_A)          # For Real Robot + UPC
robot = ysw.initialize(robotConfig.mode, robotConfig.address, robotConfig.model, option=ysw.CONNECT_AND_POWER)
if (robot != None):
    model = robot.model()

time.sleep(1)


# ==========================================================================================================
# [Gripper Initialization]
# Gripper Setting (Need to hard check of gripper ping test)
# ==========================================================================================================
gripper = ysw.Gripper()
gripper.connect_hardCheck(
    device_name=ysw.GRIPPER_DEVICE_NAME, 
    protocol_version=ysw.DXL_PROTOCOL_VERSION, 
    baudrate=ysw.DXL_BAUDRATE,
    retry_interval=1.0,
    max_retry=5
)


# ==========================================================================================================
# [Depth Camera Initialization]
# D435i:  Head
# D405_R: Right Wrist
# D405_L: Left Wrist
# ==========================================================================================================
d435i = ysw.RealSenseDepthCamera(rgb_on=True, depth_on=True, pixel=(ysw.D435I_PIXEL_WIDTH,ysw.D435I_PIXEL_HEIGHT), fps=30, serial_number=ysw.HEAD_D435I_SERIAL)
d435i.initialize()
d405_R = ysw.RealSenseDepthCamera(rgb_on=True, depth_on=True, pixel=(ysw.D405_PIXEL_WIDTH,ysw.D405_PIXEL_HEIGHT), fps=30, serial_number=ysw.RIGHT_D405_SERIAL)
d405_R.initialize()
d405_L = ysw.RealSenseDepthCamera(rgb_on=True, depth_on=True, pixel=(ysw.D405_PIXEL_WIDTH,ysw.D405_PIXEL_HEIGHT), fps=30, serial_number=ysw.LEFT_D405_SERIAL)
d405_L.initialize()

*************************** [Initialization Starts] ****************************

[INFO] SUCCESS: Connected to the Robot!
[INFO] SUCCESS: Power ON the Robot!
[INFO] SUCCESS: Servo ON the Robot!
[INFO] SUCCESS: Enable the Control Manager!
[INFO] SUCCESS: Enable the Grippers!

*************************** [Intialization is Done] ****************************



*********************** [Gripper Hard Connection Starts] ***********************

[Gripper] Trying to connect gripper... attempt=1
************************* [Gripper Connection Starts] **************************

[INFO] SUCCESS: open the port: /dev/rby1_gripper
[INFO] SUCCESS: change the baudrate 2000000

************************* [Gripper Connection is Done] *************************



************************** [Gripper Ping Test Starts] **************************

[INFO] [DXL ID:0] SUCCESS: Dynamixel model number is 1030
[INFO] [DXL ID:1] SUCCESS: Dynamixel model number is 1030

************************* [Gripper Ping Test is Do

<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Posture [Initial]
</span>

In [4]:
desired_pos = ysw.PACK_POS  
ysw.positionCtrl_joint_22dof(robot, desired_pos, minimum_time=position_control_minimum_time, control_hold_time=0.5)

******************** [22-DoF Joint Position Control Starts] ********************


********************* [22-DoF Joint Position Control Ends] *********************





True

<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Table Distance Detection [5 seconds]
</span>

In [5]:
# ============================================================
# Get the Distance of Center Pixel 
# Table distance is approximated as the depth of center pixel
# ============================================================
depth_to_table_avg = ysw.measure_depth_of_roi_center(    
    depthCamObj=d435i,
    pixel_roi_x=pixel_roi_x_tableDistance,
    pixel_roi_y=pixel_roi_y_tableDistance,
    record_duration=vision_record_duration_tableDistance,
    verbose=True
)

moved_distance_to_table = depth_to_table_avg - offset_table_distance
print(f"moved_distance_to_table: {moved_distance_to_table} [m]")

[INFO] [Head D435i] Pipeline started.
ROI Center Depth: 1.510 [m]
ROI Center Depth: 1.510 [m]
ROI Center Depth: 1.510 [m]
ROI Center Depth: 1.510 [m]
ROI Center Depth: 1.510 [m]
ROI Center Depth: 1.510 [m]
[WARNING] Invalid depth.
[WARNING] Invalid depth.
[WARNING] Invalid depth.
[WARNING] Invalid depth.
ROI Center Depth: 1.513 [m]
ROI Center Depth: 1.517 [m]
ROI Center Depth: 1.513 [m]
ROI Center Depth: 1.499 [m]
ROI Center Depth: 1.521 [m]
ROI Center Depth: 1.513 [m]
ROI Center Depth: 1.532 [m]
ROI Center Depth: 1.525 [m]
ROI Center Depth: 1.528 [m]
ROI Center Depth: 1.532 [m]
ROI Center Depth: 1.521 [m]
ROI Center Depth: 1.506 [m]
ROI Center Depth: 1.513 [m]
ROI Center Depth: 1.506 [m]
ROI Center Depth: 1.521 [m]
ROI Center Depth: 1.502 [m]
ROI Center Depth: 1.510 [m]
ROI Center Depth: 1.517 [m]
ROI Center Depth: 1.521 [m]
ROI Center Depth: 1.513 [m]
ROI Center Depth: 1.513 [m]
ROI Center Depth: 1.517 [m]
ROI Center Depth: 1.521 [m]
ROI Center Depth: 1.521 [m]
ROI Center Depth: 1.51

<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Posture [DEMO POSE]
</span>

In [6]:
desired_pos = ysw.DEMO_BASIC_POS_260826  
ysw.positionCtrl_joint_22dof(robot, desired_pos, minimum_time=position_control_minimum_time, control_hold_time=0.5)

******************** [22-DoF Joint Position Control Starts] ********************


********************* [22-DoF Joint Position Control Ends] *********************





True

<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Move to Table
</span>

In [7]:
ysw.mobile_linear_distance(
    robot,
    distance_m=moved_distance_to_table,
    max_speed=0.25,
    accel=0.8,
    tolerance_m=0.01
)

distance=0.000[m], speed=0.016[m/s], pos_error=1.117[m], yaw_error=0.00[deg]
distance=0.010[m], speed=0.144[m/s], pos_error=1.107[m], yaw_error=-0.04[deg]
distance=0.050[m], speed=0.250[m/s], pos_error=1.067[m], yaw_error=0.01[deg]
distance=0.102[m], speed=0.250[m/s], pos_error=1.015[m], yaw_error=0.01[deg]
distance=0.153[m], speed=0.250[m/s], pos_error=0.963[m], yaw_error=0.01[deg]
distance=0.206[m], speed=0.250[m/s], pos_error=0.911[m], yaw_error=-0.01[deg]
distance=0.258[m], speed=0.250[m/s], pos_error=0.859[m], yaw_error=0.02[deg]
distance=0.311[m], speed=0.250[m/s], pos_error=0.806[m], yaw_error=-0.00[deg]
distance=0.363[m], speed=0.250[m/s], pos_error=0.754[m], yaw_error=0.00[deg]
distance=0.415[m], speed=0.250[m/s], pos_error=0.702[m], yaw_error=-0.01[deg]
distance=0.468[m], speed=0.250[m/s], pos_error=0.649[m], yaw_error=-0.01[deg]
distance=0.519[m], speed=0.250[m/s], pos_error=0.597[m], yaw_error=-0.02[deg]
distance=0.571[m], speed=0.250[m/s], pos_error=0.545[m], yaw_error=-0.

True

<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Check [Black Box] [5 seconds]
</span>

In [8]:
# ============================================================
# Black Box check
# ============================================================
result_blackBox = fn.check_colored_box_and_set_grasp_arm(
    depthCamObj=d435i,
    boxColor="black",
    hsv_mask=ysw.HSV_MASK_BLACK_BOX,
    record_duration=vision_record_duration_checkBox,
    box_minimum_area=box_minimum_area,
    box_maximum_distance=box_maximum_distance,
    pixel_roi_x=pixel_roi_x_tableTop,
    pixel_roi_y=pixel_roi_y_tableTop,
    verbose=True,
)

blackBox_samples = result_blackBox["box_samples"]
blackBox_sample_avg = result_blackBox["box_sample_avg"]
grasp_arm_blackObj = result_blackBox["grasp_arm"]

[INFO] [Head D435i] Pipeline started.
[INFO] [Head D435i] Pipeline stopped.

========== Black Box Detection Result ==========

[Black Box]
num_samples             : 95
area_2d_avg             : 16002.3
area_2d_median          : 15898.5
center_3d_avg           : [-0.2192  0.0322  0.7304] m
center_3d_median        : [-0.2192  0.0321  0.731 ] m
bbox_3d_size_avg        : [0.1233 0.174  0.1631] m
bbox_3d_size_median     : [0.1204 0.1722 0.155 ] m

grasp_arm_blackObj: left


<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Check [Black Objects] [5 seconds]
</span>

In [9]:
result_blackObj = fn.check_colored_objects_sorted_by_center(
    depthCamObj=d435i,
    objectColor="black",
    hsv_mask=ysw.HSV_MASK_BLACK_BOX,
    record_duration=vision_record_duration_checkObject,
    object_minimum_area=object_minimum_area,
    object_maximum_area=object_maximum_area,
    maximum_depth=object_maximum_distance,
    pixel_roi_x=pixel_roi_x_tableTop,
    pixel_roi_y=pixel_roi_y_tableTop,
    max_match_distance_pixel=40.0,
    min_rank_sample_ratio=0.4,
    min_rank_num_samples=20,
    verbose=True,
)

blackObj_sample_avg_list = result_blackObj["object_sample_avg_list"]

[INFO] [Head D435i] Pipeline started.
[INFO] [Head D435i] Pipeline stopped.

========== Black Object Detection Result ==========
Objects are clustered by pixel_center after recording.
Final ranks are sorted by distance from image/ROI center.

Total raw detections     : 551
Total clusters           : 7
Valid clusters           : 4
Ignored clusters         : 3
Max sample count         : 135
Cluster distance thresh  : 40.0 pixel
Sample filter            : ratio >= 0.4, num_samples >= 20

[Black Object rank 0]
cluster_idx                  : 3
num_samples                  : 135
sample_ratio                 : 1.00
area_2d_avg                  : 563.3
area_2d_median               : 558.0
center_distance_pixel_avg    : 17.5
center_distance_pixel_median : 17.5
pixel_center_mean            : [337.1 294.1]
pixel_center_median          : [337. 294.]
center_3d_avg                : [0.0347 0.0586 0.7297] m
center_3d_median             : [0.0347 0.0585 0.7295] m
bbox_3d_size_avg             : [0.0262

<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
IK for [Black Objects and Box]
</span>

In [10]:
black_target_result_list = []

for obj_idx, blackObj_sample_avg in enumerate(blackObj_sample_avg_list):
    if blackObj_sample_avg is None:
        print(f"[Black Object {obj_idx}] skipped: sample_avg is None")
        black_target_result_list.append(None)
        continue
    
    print(f"\n# ==================================================")
    print(f"# Compute IK for Black Object {obj_idx}")
    print(f"# ==================================================")
    
    black_target_result = fn.compute_IK_for_colored_object_and_box(
        robot=robot,
        objectColor=f"black_{obj_idx}",
        object_sample_avg=blackObj_sample_avg,
        boxColor="black",
        box_sample_avg=blackBox_sample_avg,
        grasp_arm=grasp_arm_blackObj,
        object_manual_offset_downToGrasp=object_manual_offset_downToGrasp,
        object_pregrasp_z_offset=object_pregrasp_z_offset,
        object_grasp_x_offset=object_grasp_x_offset, 
        box_pregrasp_z_offset=box_pregrasp_z_offset,
        box_grasp_x_offset=box_grasp_x_offset, 
        box_grasp_y_offset=box_grasp_y_offset,
        box_grasp_z_offset=box_grasp_z_offset,
        use_median=False,
        gripper_z_offset=ysw.GRIPPER_Z_OFFSET_FROM_LINK_EE_FINGER,
        verbose=True,
    )
    
    black_target_result_list.append(black_target_result)


# ==================================================
# Compute IK for Black Object 0
# ==================================================
******************* [FK of 'link_head_2' from 'base' Starts] *******************

[Result]

T:
[[ 0.7314 -0.      0.682   0.022 ]
 [ 0.      1.      0.      0.    ]
 [-0.682  -0.      0.7314  1.4244]
 [ 0.      0.      0.      1.    ]]


position [m]: [0.022  0.     1.4244]
rpy [rad]: [-0.      0.7505  0.    ]
rpy [deg]: [-0.     43.      0.0002]

******************** [FK of 'link_head_2' from 'base' Ends] ********************



======== For Black_0 Object ========
Before open gripper: 
[[ 0.      0.766  -0.6428  0.5245]
 [-1.      0.     -0.     -0.0182]
 [ 0.      0.6428  0.766   1.0815]
 [ 0.      0.      0.      1.    ]]

After open gripper: 
[[ 0.      0.766  -0.6428  0.5395]
 [-1.      0.     -0.     -0.0182]
 [ 0.      0.6428  0.766   0.9765]
 [ 0.      0.      0.      1.    ]]

Grasping Arm: left

======== For Black Box ========
For Black B

<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Pick [Black objects] from the center and Place to [Black Box]
</span>

In [11]:
for obj_idx, black_target_result in enumerate(black_target_result_list):
    if black_target_result is None:
        print(f"[Black Object {obj_idx}] skipped: target_result is None")
        continue

    print(f"\n# ==================================================")
    print(f"# Pick and Place Black Object {obj_idx}")
    print(f"# ==================================================")

    rv = fn.pick_object_and_place_to_box(
        robot=robot,
        gripper=gripper,
        target_result=black_target_result,
        offset_gripper_open_m=offset_gripper_open_m,
        grasp_current_obj=grasp_current_obj,
        object_floating_z_afterGrasp=object_floating_z,
        position_control_minimum_time=position_control_minimum_time,
        control_hold_time=0.5,
        fix_torso=True,
        verbose=True,
    )

    if rv is False:
        print(f"[Black Object {obj_idx}] Pick-place failed.")
        break


# ==================================================
# Pick and Place Black Object 0
# ==================================================

# ======================================
# Move to Pre-grasp pose: black_0
# ======================================
***************************** [Robot State Starts] *****************************


****************************** [Robot State Ends] ******************************



*************** [Cartesian Position Control(Whole-Body) Starts] ****************

[INFO] Add (Whole-Body) Cartesian target 0: 'base'->'ee_finger_l1', pos=[ 0.5245 -0.0182  1.0815], rpy=[ 0.6981 -0.     -1.5708]
[INFO] Add joint position target: torso_0 = 0.0
[INFO] Add joint position target: torso_1 = 0.4363302035640556
[INFO] Add joint position target: torso_2 = -0.8726642041102595
[INFO] Add joint position target: torso_3 = 0.43633400054620386

**************** [Cartesian Position Control(Whole-Body) Ends] *****************




# ======================================


<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Posture [DEMO POSE]
</span>

In [12]:
desired_pos = ysw.DEMO_BASIC_POS_260826  
ysw.positionCtrl_joint_22dof(robot, desired_pos, minimum_time=position_control_minimum_time, control_hold_time=0.5)

******************** [22-DoF Joint Position Control Starts] ********************


********************* [22-DoF Joint Position Control Ends] *********************





True

<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Check [Red Box] [5 seconds]
</span>

In [13]:
# ============================================================
# Red Box check
# ============================================================
result_redBox = fn.check_colored_box_and_set_grasp_arm(
    depthCamObj=d435i,
    boxColor="red",
    hsv_mask=ysw.HSV_MASK_RED_BOX,
    record_duration=vision_record_duration_checkBox,
    box_minimum_area=box_minimum_area,
    box_maximum_distance=box_maximum_distance,
    pixel_roi_x=pixel_roi_x_tableTop,
    pixel_roi_y=pixel_roi_y_tableTop,
    verbose=True,
)

redBox_samples = result_redBox["box_samples"]
redBox_sample_avg = result_redBox["box_sample_avg"]
grasp_arm_redObj = result_redBox["grasp_arm"]

[INFO] [Head D435i] Pipeline started.
[INFO] [Head D435i] Pipeline stopped.

========== Red Box Detection Result ==========

[Red Box]
num_samples             : 95
area_2d_avg             : 13765.7
area_2d_median          : 13571.5
center_3d_avg           : [0.2337 0.0221 0.7454] m
center_3d_median        : [0.2335 0.0227 0.746 ] m
bbox_3d_size_avg        : [0.158  0.1814 0.2443] m
bbox_3d_size_median     : [0.157  0.1822 0.245 ] m

grasp_arm_redObj: right


<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Check [Red Objects] [5 seconds]
</span>

In [14]:
result_redObj = fn.check_colored_objects_sorted_by_center(
    depthCamObj=d435i,
    objectColor="red",
    hsv_mask=ysw.HSV_MASK_RED,
    record_duration=vision_record_duration_checkObject,
    object_minimum_area=object_minimum_area,
    object_maximum_area=object_maximum_area,
    maximum_depth=object_maximum_distance,
    pixel_roi_x=pixel_roi_x_tableTop,
    pixel_roi_y=pixel_roi_y_tableTop,
    max_match_distance_pixel=40.0,
    min_rank_sample_ratio=0.4,
    min_rank_num_samples=20,
    verbose=True,
)

redObj_sample_avg_list = result_redObj["object_sample_avg_list"]

[INFO] [Head D435i] Pipeline started.
[INFO] [Head D435i] Pipeline stopped.

========== Red Object Detection Result ==========
Objects are clustered by pixel_center after recording.
Final ranks are sorted by distance from image/ROI center.

Total raw detections     : 519
Total clusters           : 4
Valid clusters           : 4
Ignored clusters         : 0
Max sample count         : 133
Cluster distance thresh  : 40.0 pixel
Sample filter            : ratio >= 0.4, num_samples >= 20

[Red Object rank 0]
cluster_idx                  : 0
num_samples                  : 132
sample_ratio                 : 0.99
area_2d_avg                  : 595.3
area_2d_median               : 673.5
center_distance_pixel_avg    : 37.9
center_distance_pixel_median : 38.2
pixel_center_mean            : [282.5 294.7]
pixel_center_median          : [282. 294.]
center_3d_avg                : [-0.0313  0.0584  0.7253] m
center_3d_median             : [-0.0321  0.0577  0.726 ] m
bbox_3d_size_avg             : [0.03

<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
IK for [Red Objects and Box]
</span>

In [15]:
red_target_result_list = []

for obj_idx, redObj_sample_avg in enumerate(redObj_sample_avg_list):
    if redObj_sample_avg is None:
        print(f"[Red Object {obj_idx}] skipped: sample_avg is None")
        red_target_result_list.append(None)
        continue
    
    print(f"\n# ==================================================")
    print(f"# Compute IK for Red Object {obj_idx}")
    print(f"# ==================================================")
    
    red_target_result = fn.compute_IK_for_colored_object_and_box(
        robot=robot,
        objectColor=f"red_{obj_idx}",
        object_sample_avg=redObj_sample_avg,
        boxColor="red",
        box_sample_avg=redBox_sample_avg,
        grasp_arm=grasp_arm_redObj,
        object_manual_offset_downToGrasp=object_manual_offset_downToGrasp,
        object_pregrasp_z_offset=object_pregrasp_z_offset,
        object_grasp_x_offset=object_grasp_x_offset, 
        box_pregrasp_z_offset=box_pregrasp_z_offset,
        box_grasp_x_offset=box_grasp_x_offset,
        box_grasp_y_offset=box_grasp_y_offset,
        box_grasp_z_offset=box_grasp_z_offset - 0.008,       # Manual
        use_median=False,
        gripper_z_offset=ysw.GRIPPER_Z_OFFSET_FROM_LINK_EE_FINGER,
        verbose=True,
    )
    red_target_result_list.append(red_target_result)


# ==================================================
# Compute IK for Red Object 0
# ==================================================
******************* [FK of 'link_head_2' from 'base' Starts] *******************

[Result]

T:
[[ 0.7314 -0.      0.682   0.022 ]
 [-0.      1.      0.      0.    ]
 [-0.682  -0.      0.7314  1.4244]
 [ 0.      0.      0.      1.    ]]


position [m]: [0.022  0.     1.4244]
rpy [rad]: [-0.      0.7505 -0.    ]
rpy [deg]: [-0.0003 43.0003 -0.0002]

******************** [FK of 'link_head_2' from 'base' Ends] ********************



======== For Red_0 Object ========
Before open gripper: 
[[ 0.     -0.766  -0.6428  0.5215]
 [ 1.      0.      0.      0.0478]
 [ 0.     -0.6428  0.766   1.0846]
 [ 0.      0.      0.      1.    ]]

After open gripper: 
[[ 0.     -0.766  -0.6428  0.5365]
 [ 1.      0.      0.      0.0478]
 [ 0.     -0.6428  0.766   0.9796]
 [ 0.      0.      0.      1.    ]]

Grasping Arm: right

======== For Red Box ========
For Red Box: 
[[

<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Pick [Red objects] from the center and Place to [Red Box]
</span>

In [16]:
for obj_idx, red_target_result in enumerate(red_target_result_list):
    if red_target_result is None:
        print(f"[Red Object {obj_idx}] skipped: target_result is None")
        continue

    print(f"\n# ==================================================")
    print(f"# Pick and Place Red Object {obj_idx}")
    print(f"# ==================================================")

    rv = fn.pick_object_and_place_to_box(
        robot=robot,
        gripper=gripper,
        target_result=red_target_result,
        offset_gripper_open_m=offset_gripper_open_m,
        grasp_current_obj=grasp_current_obj,
        object_floating_z_afterGrasp=object_floating_z,
        position_control_minimum_time=position_control_minimum_time,
        control_hold_time=0.5,
        fix_torso=True,
        verbose=True,
    )

    if rv is False:
        print(f"[Red Object {obj_idx}] Pick-place failed.")
        break


# ==================================================
# Pick and Place Red Object 0
# ==================================================

# ======================================
# Move to Pre-grasp pose: red_0
# ======================================
***************************** [Robot State Starts] *****************************


****************************** [Robot State Ends] ******************************



*************** [Cartesian Position Control(Whole-Body) Starts] ****************

[INFO] Add (Whole-Body) Cartesian target 0: 'base'->'ee_finger_r1', pos=[0.5215 0.0478 1.0846], rpy=[-0.6981 -0.      1.5708]
[INFO] Add joint position target: torso_0 = 0.0
[INFO] Add joint position target: torso_1 = 0.4363302035640556
[INFO] Add joint position target: torso_2 = -0.8726604071281112
[INFO] Add joint position target: torso_3 = 0.43633400054620386

**************** [Cartesian Position Control(Whole-Body) Ends] *****************




# ======================================
# Open 

<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Posture [DEMO POSE]
</span>

In [17]:
desired_pos = ysw.DEMO_BASIC_POS_260826  
ysw.positionCtrl_joint_22dof(robot, desired_pos, minimum_time=position_control_minimum_time * 0.4, control_hold_time=0.5)

******************** [22-DoF Joint Position Control Starts] ********************


********************* [22-DoF Joint Position Control Ends] *********************





True

<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Grasp Black and Red Boxes
</span>

In [18]:
# Black Box Grasp
rv = fn.grasp_box_outerside_and_lift(
    robot=robot,
    gripper=gripper,
    target_result=black_target_result_list[0],
    box_grasp_current=grasp_current_box,
    box_grasp_gripper_open_pos_mm=box_grasp_gripper_open_pos_mm,
    box_floating_z=box_grasp_floating_offset,
    position_control_minimum_time=position_control_minimum_time,
    control_hold_time=0.5,
    fix_torso=True,
    verbose=True,
)

# Red Box Grasp
rv = fn.grasp_box_outerside_and_lift(
    robot=robot,
    gripper=gripper,
    target_result=red_target_result_list[0],
    box_grasp_current=grasp_current_box,
    box_grasp_gripper_open_pos_mm=box_grasp_gripper_open_pos_mm,
    box_floating_z=box_grasp_floating_offset,
    position_control_minimum_time=position_control_minimum_time,
    control_hold_time=0.5,
    fix_torso=True,
    verbose=True,
)


# ==================================================
# Move to black Box Side Before Opening Gripper
# ==================================================
***************************** [Robot State Starts] *****************************


****************************** [Robot State Ends] ******************************



*************** [Cartesian Position Control(Whole-Body) Starts] ****************

[INFO] Add (Whole-Body) Cartesian target 0: 'base'->'ee_finger_l1', pos=[0.5031 0.2473 1.1372], rpy=[ 0.6981 -0.     -1.5708]
[INFO] Add joint position target: torso_0 = 7.593964296463571e-06
[INFO] Add joint position target: torso_1 = 0.43632640658190736
[INFO] Add joint position target: torso_2 = -0.8726717980745559
[INFO] Add joint position target: torso_3 = 0.43633400054620386

**************** [Cartesian Position Control(Whole-Body) Ends] *****************




# ==================================================
# Open Gripper for grasping black Box
# =================================

<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Set Box-Move Ready Posture
</span>

In [19]:
desired_pos = ysw.DEMO_MOVE_POS_260826  
ysw.positionCtrl_joint_22dof(robot, desired_pos, minimum_time=position_control_minimum_time * 0.4, control_hold_time=0.5)

******************** [22-DoF Joint Position Control Starts] ********************


********************* [22-DoF Joint Position Control Ends] *********************





True

In [20]:
# For Black Box
rv = fn.move_ee_to_box_lift_ready_pose(
    robot=robot,
    target_result=black_target_result_list[0],
    pos_right=box_move_readyPosition_right,      # manual setting
    pos_left=box_move_readyPosition_left,        # manual setting
    target_T_key="T_SC_target_for_box_side_after_open_gripper",
    minimum_time=position_control_minimum_time * 0.7,
    control_hold_time=0.5,
    get_return=True,
)

# For Red Box
rv = fn.move_ee_to_box_lift_ready_pose(
    robot=robot,
    target_result=red_target_result_list[0],
    pos_right=box_move_readyPosition_right,      # manual setting
    pos_left=box_move_readyPosition_left,        # manual setting
    target_T_key="T_SC_target_for_box_side_after_open_gripper",
    minimum_time=position_control_minimum_time * 0.7,
    control_hold_time=0.5,
    get_return=True,
)

*************** [Cartesian Position Control(Whole-Body) Starts] ****************

[INFO] Add (Whole-Body) Cartesian target 0: 'base'->'ee_finger_l1', pos=[0.55 0.25 1.2 ], rpy=[ 0.6981 -0.     -1.5708]
[INFO] Add joint position target: torso_0 = 0.0
[INFO] Add joint position target: torso_1 = 0.0
[INFO] Add joint position target: torso_2 = 0.0
[INFO] Add joint position target: torso_3 = 0.0
[INFO] Add joint position target: torso_4 = 0.0
[INFO] Add joint position target: torso_5 = 0.0

**************** [Cartesian Position Control(Whole-Body) Ends] *****************



*************** [Cartesian Position Control(Whole-Body) Starts] ****************

[INFO] Add (Whole-Body) Cartesian target 0: 'base'->'ee_finger_r1', pos=[ 0.55 -0.25  1.2 ], rpy=[-0.6981 -0.      1.5708]
[INFO] Add joint position target: torso_0 = 0.0
[INFO] Add joint position target: torso_1 = 0.0
[INFO] Add joint position target: torso_2 = 0.0
[INFO] Add joint position target: torso_3 = 0.0
[INFO] Add joint position ta

<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Move Backward slightly
</span>

In [21]:
ysw.mobile_linear_distance(
    robot,
    distance_m=moved_distance_to_back,
    max_speed=0.12,
    accel=0.1,
    tolerance_m=0.01
)
time.sleep(1)

distance=0.000[m], speed=-0.002[m/s], pos_error=0.700[m], yaw_error=0.00[deg]
distance=-0.001[m], speed=-0.018[m/s], pos_error=0.699[m], yaw_error=0.00[deg]
distance=-0.006[m], speed=-0.034[m/s], pos_error=0.694[m], yaw_error=-0.01[deg]
distance=-0.013[m], speed=-0.050[m/s], pos_error=0.687[m], yaw_error=-0.01[deg]
distance=-0.025[m], speed=-0.066[m/s], pos_error=0.675[m], yaw_error=-0.01[deg]
distance=-0.040[m], speed=-0.082[m/s], pos_error=0.660[m], yaw_error=0.01[deg]
distance=-0.058[m], speed=-0.098[m/s], pos_error=0.642[m], yaw_error=-0.01[deg]
distance=-0.079[m], speed=-0.114[m/s], pos_error=0.621[m], yaw_error=-0.01[deg]
distance=-0.104[m], speed=-0.120[m/s], pos_error=0.596[m], yaw_error=-0.01[deg]
distance=-0.129[m], speed=-0.120[m/s], pos_error=0.571[m], yaw_error=-0.03[deg]
distance=-0.153[m], speed=-0.120[m/s], pos_error=0.547[m], yaw_error=-0.02[deg]
distance=-0.178[m], speed=-0.120[m/s], pos_error=0.522[m], yaw_error=-0.01[deg]
distance=-0.203[m], speed=-0.120[m/s], pos_e

<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Turn 180 degree
</span>

In [22]:
ysw.mobile_turn_inplace(
    robot,
    angle_deg=desired_yaw_deg_to_turn, 
    max_yaw_vel=0.3,
    yaw_accel=1.5,
    tolerance_deg=1.0
)

yaw_angle=0.00[deg], yaw_velocity=-0.030[rad/s], yaw_error=176.00[deg]
yaw_angle=-0.81[deg], yaw_velocity=-0.270[rad/s], yaw_error=175.19[deg]
yaw_angle=-4.60[deg], yaw_velocity=-0.300[rad/s], yaw_error=171.40[deg]
yaw_angle=-8.17[deg], yaw_velocity=-0.300[rad/s], yaw_error=167.83[deg]
yaw_angle=-11.73[deg], yaw_velocity=-0.300[rad/s], yaw_error=164.27[deg]
yaw_angle=-15.29[deg], yaw_velocity=-0.300[rad/s], yaw_error=160.71[deg]
yaw_angle=-18.85[deg], yaw_velocity=-0.300[rad/s], yaw_error=157.15[deg]
yaw_angle=-22.42[deg], yaw_velocity=-0.300[rad/s], yaw_error=153.58[deg]
yaw_angle=-25.89[deg], yaw_velocity=-0.300[rad/s], yaw_error=150.11[deg]
yaw_angle=-29.47[deg], yaw_velocity=-0.300[rad/s], yaw_error=146.53[deg]
yaw_angle=-33.10[deg], yaw_velocity=-0.300[rad/s], yaw_error=142.90[deg]
yaw_angle=-36.63[deg], yaw_velocity=-0.300[rad/s], yaw_error=139.37[deg]
yaw_angle=-40.13[deg], yaw_velocity=-0.300[rad/s], yaw_error=135.87[deg]
yaw_angle=-43.67[deg], yaw_velocity=-0.300[rad/s], yaw_e

True

<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Shelf Distance Detection [5 seconds]
</span>

In [23]:
shelf_distance_result = fn.measure_distance_by_nearest_depth_in_roi(
    depthCamObj=d435i,
    pixel_roi_x=pixel_roi_x_shelfDistance,
    pixel_roi_y=pixel_roi_y_shelfDistance,
    record_duration=vision_record_duration_shelfDistance,
    nearest_ratio=0.05,      # 가장 가까운 5%
    min_num_points=30,       # 최소 30개 pixel 평균
    max_valid_depth=3.0,     # 필요하면 2.0 [m] 이런 식으로 제한 가능
)
moved_distance_to_shelf = shelf_distance_result["distance_avg"] - offset_shelf_distance
print(f"moved_distance_to_shelf: {moved_distance_to_shelf:.4f} [m]")

[INFO] [Head D435i] Pipeline started.
Distance nearest_avg: 2.106 [m], median: 2.117 [m], min: 2.033 [m], used: 3985, valid: 79706
Distance nearest_avg: 2.106 [m], median: 2.117 [m], min: 2.033 [m], used: 3985, valid: 79706
Distance nearest_avg: 2.106 [m], median: 2.117 [m], min: 2.033 [m], used: 3985, valid: 79706
Distance nearest_avg: 2.115 [m], median: 2.117 [m], min: 2.060 [m], used: 4067, valid: 81340
Distance nearest_avg: 2.120 [m], median: 2.124 [m], min: 2.088 [m], used: 4077, valid: 81552
Distance nearest_avg: 2.097 [m], median: 2.102 [m], min: 2.006 [m], used: 4161, valid: 83232
Distance nearest_avg: 2.082 [m], median: 2.088 [m], min: 2.006 [m], used: 4226, valid: 84522
Distance nearest_avg: 2.083 [m], median: 2.088 [m], min: 2.013 [m], used: 4238, valid: 84778
Distance nearest_avg: 2.067 [m], median: 2.081 [m], min: 1.974 [m], used: 4177, valid: 83544
Distance nearest_avg: 2.098 [m], median: 2.102 [m], min: 2.033 [m], used: 4253, valid: 85064
Distance nearest_avg: 2.088 [m],

<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Move Forward to Shelf
</span>

In [24]:
ysw.mobile_linear_distance(
    robot,
    distance_m=moved_distance_to_shelf,
    max_speed=0.25,
    accel=0.8,
    tolerance_m=0.01
)

distance=0.000[m], speed=0.016[m/s], pos_error=1.614[m], yaw_error=0.00[deg]
distance=0.010[m], speed=0.144[m/s], pos_error=1.604[m], yaw_error=0.02[deg]
distance=0.049[m], speed=0.250[m/s], pos_error=1.565[m], yaw_error=-0.10[deg]
distance=0.102[m], speed=0.250[m/s], pos_error=1.512[m], yaw_error=-0.00[deg]
distance=0.154[m], speed=0.250[m/s], pos_error=1.461[m], yaw_error=0.05[deg]
distance=0.207[m], speed=0.250[m/s], pos_error=1.407[m], yaw_error=0.02[deg]
distance=0.260[m], speed=0.250[m/s], pos_error=1.355[m], yaw_error=0.03[deg]
distance=0.312[m], speed=0.250[m/s], pos_error=1.303[m], yaw_error=0.01[deg]
distance=0.364[m], speed=0.250[m/s], pos_error=1.251[m], yaw_error=0.01[deg]
distance=0.417[m], speed=0.250[m/s], pos_error=1.197[m], yaw_error=0.01[deg]
distance=0.469[m], speed=0.250[m/s], pos_error=1.145[m], yaw_error=0.00[deg]
distance=0.522[m], speed=0.250[m/s], pos_error=1.093[m], yaw_error=0.01[deg]
distance=0.573[m], speed=0.250[m/s], pos_error=1.041[m], yaw_error=0.01[de

True

<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Open Gripper(Lay Boxes) and Close Gripper
</span>

In [25]:
gripper.initialize_setting(dxl_id=ysw.GRIPPER_ID_L, operation_mode=ysw.DXL_CURRENT_CONTROL_MODE)
gripper.write_current_ampere(dxl_id=ysw.GRIPPER_ID_L, current_ampere=gripper_open_ampere)
time.sleep(1)

gripper.initialize_setting(dxl_id=ysw.GRIPPER_ID_R, operation_mode=ysw.DXL_CURRENT_CONTROL_MODE)
gripper.write_current_ampere(dxl_id=ysw.GRIPPER_ID_R, current_ampere=gripper_open_ampere)
time.sleep(1)

gripper.initialize_setting(dxl_id=ysw.GRIPPER_ID_L, operation_mode=ysw.DXL_CURRENT_CONTROL_MODE)
gripper.write_current_ampere(dxl_id=ysw.GRIPPER_ID_L, current_ampere=gripper_close_ampere)
time.sleep(1)

gripper.initialize_setting(dxl_id=ysw.GRIPPER_ID_R, operation_mode=ysw.DXL_CURRENT_CONTROL_MODE)
gripper.write_current_ampere(dxl_id=ysw.GRIPPER_ID_R, current_ampere=gripper_close_ampere)
time.sleep(1)

********************* [Gripper [DXL ID:1] setting Starts] **********************

[INFO] SUCCESS: [DXL ID:1] Torque is Disabled
[INFO] SUCCESS: [DXL ID:1] Set Current Control Mode
[INFO] SUCCESS: [DXL ID:1] Torque is Enabled

********************* [Gripper [DXL ID:1] setting is Done] *********************



[INFO] SUCCESS: [DXL ID:1] Set Current [-0.1 A]
********************* [Gripper [DXL ID:0] setting Starts] **********************

[INFO] SUCCESS: [DXL ID:0] Torque is Disabled
[INFO] SUCCESS: [DXL ID:0] Set Current Control Mode
[INFO] SUCCESS: [DXL ID:0] Torque is Enabled

********************* [Gripper [DXL ID:0] setting is Done] *********************



[INFO] SUCCESS: [DXL ID:0] Set Current [-0.1 A]
********************* [Gripper [DXL ID:1] setting Starts] **********************

[INFO] SUCCESS: [DXL ID:1] Torque is Disabled
[INFO] SUCCESS: [DXL ID:1] Set Current Control Mode
[INFO] SUCCESS: [DXL ID:1] Torque is Enabled

********************* [Gripper [DXL ID:1] setting is Done]

<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Move Backward slightly
</span>

In [26]:
ysw.mobile_linear_distance(
    robot,
    distance_m=moved_distance_to_back,
    max_speed=0.25,
    accel=0.8,
    tolerance_m=0.01
)
time.sleep(1)

distance=0.000[m], speed=-0.016[m/s], pos_error=0.700[m], yaw_error=0.00[deg]
distance=-0.010[m], speed=-0.144[m/s], pos_error=0.690[m], yaw_error=-0.01[deg]
distance=-0.050[m], speed=-0.250[m/s], pos_error=0.650[m], yaw_error=-0.03[deg]
distance=-0.103[m], speed=-0.250[m/s], pos_error=0.597[m], yaw_error=-0.08[deg]
distance=-0.155[m], speed=-0.250[m/s], pos_error=0.545[m], yaw_error=-0.05[deg]
distance=-0.208[m], speed=-0.250[m/s], pos_error=0.492[m], yaw_error=0.07[deg]
distance=-0.260[m], speed=-0.250[m/s], pos_error=0.440[m], yaw_error=-0.09[deg]
distance=-0.311[m], speed=-0.250[m/s], pos_error=0.389[m], yaw_error=0.02[deg]
distance=-0.363[m], speed=-0.250[m/s], pos_error=0.337[m], yaw_error=0.07[deg]
distance=-0.415[m], speed=-0.250[m/s], pos_error=0.285[m], yaw_error=-0.00[deg]
distance=-0.466[m], speed=-0.250[m/s], pos_error=0.234[m], yaw_error=0.00[deg]
distance=-0.518[m], speed=-0.250[m/s], pos_error=0.182[m], yaw_error=-0.01[deg]
distance=-0.570[m], speed=-0.250[m/s], pos_err

<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Turn 180 degree
</span>

In [27]:

ysw.mobile_turn_inplace(
    robot,
    angle_deg=(-1.0)*desired_yaw_deg_to_turn, 
    max_yaw_vel=0.3,
    yaw_accel=1.5,
    tolerance_deg=1.0
)
time.sleep(1)

yaw_angle=0.00[deg], yaw_velocity=0.030[rad/s], yaw_error=176.00[deg]
yaw_angle=0.87[deg], yaw_velocity=0.270[rad/s], yaw_error=175.13[deg]
yaw_angle=4.49[deg], yaw_velocity=0.300[rad/s], yaw_error=171.51[deg]
yaw_angle=8.14[deg], yaw_velocity=0.300[rad/s], yaw_error=167.86[deg]
yaw_angle=11.81[deg], yaw_velocity=0.300[rad/s], yaw_error=164.19[deg]
yaw_angle=15.30[deg], yaw_velocity=0.300[rad/s], yaw_error=160.70[deg]
yaw_angle=18.82[deg], yaw_velocity=0.300[rad/s], yaw_error=157.18[deg]
yaw_angle=22.29[deg], yaw_velocity=0.300[rad/s], yaw_error=153.71[deg]
yaw_angle=25.80[deg], yaw_velocity=0.300[rad/s], yaw_error=150.20[deg]
yaw_angle=29.31[deg], yaw_velocity=0.300[rad/s], yaw_error=146.69[deg]
yaw_angle=32.87[deg], yaw_velocity=0.300[rad/s], yaw_error=143.13[deg]
yaw_angle=36.34[deg], yaw_velocity=0.300[rad/s], yaw_error=139.66[deg]
yaw_angle=39.88[deg], yaw_velocity=0.300[rad/s], yaw_error=136.12[deg]
yaw_angle=43.42[deg], yaw_velocity=0.300[rad/s], yaw_error=132.58[deg]
yaw_angle=

<span style="
    color:#C9961B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Go Pack Pose
</span>

In [28]:
desired_pos = ysw.PACK_POS  
ysw.positionCtrl_joint_22dof(robot, desired_pos, minimum_time=position_control_minimum_time, control_hold_time=0.5)

******************** [22-DoF Joint Position Control Starts] ********************


********************* [22-DoF Joint Position Control Ends] *********************





True

<span style="
    color:gray;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Test Depth Camera with ROI: Show Image 
</span>

In [29]:
# # --- Select camera location ----------------------------
# location = "head"       # head / wrist_R / wrist_L
# # -------------------------------------------------------

# if location == "head":
#     cam = d435i
# elif location == "wrist_R":
#     cam = d405_R
# elif location == "wrist_L":
#     cam = d405_L
    

# # ============================================================
# # ROI setting 
# # Pixel Origin: (0,0) = 왼쪽 위 꼭짓점
# # Posture: PACK_POS
# # ============================================================
# # Case 1: Move to Table
# # pixel_roi_x = pixel_roi_x_tableDistance    
# # pixel_roi_y = pixel_roi_y_tableDistance

# # Case 2: See Table Top
# pixel_roi_x = pixel_roi_x_tableTop
# pixel_roi_y = pixel_roi_y_tableTop
    
# # ============================================================
# # Get the Distance of Center Pixel 
# # Table distance is approximated as the depth of center pixel
# # ============================================================

# cam.startPipeline()

# try:
#     while True:
#         color_frame, depth_frame = cam.getRawFrames()

#         if color_frame is None or depth_frame is None:
#             continue

#         color_image, depth_image = cam.frameToImage(color_frame, depth_frame)
#         depth_colorMap = cam.convertDepthToColorMap(depth_image, alpha=0.03)
        
#         h, w, _ = color_image.shape
        
#         # ROI center pixel in full image coordinate
#         if pixel_roi_x is None:
#             x_min = 0
#             x_max = w
#         else:
#             x_min, x_max = pixel_roi_x
#             x_min = int(np.clip(x_min, 0, w - 1))
#             x_max = int(np.clip(x_max, 0, w))

#         if pixel_roi_y is None:
#             y_min = 0
#             y_max = h
#         else:
#             y_min, y_max = pixel_roi_y
#             y_min = int(np.clip(y_min, 0, h - 1))
#             y_max = int(np.clip(y_max, 0, h))
            
#         x_roi_center = (x_min + x_max) // 2
#         y_roi_center = (y_min + y_max) // 2

#         distance = d435i.getDepth(depth_frame, (x_roi_center, y_roi_center))   # unit = [m]
        
        
#         # Draw ROI box on full image
#         cv2.rectangle(
#             color_image,
#             (x_min, y_min),
#             (x_max, y_max),
#             (255, 0, 0),
#             2
#         )
#         cv2.putText(
#             color_image,                   
#             f"ROI Box",
#             (20, 80),
#             cv2.FONT_HERSHEY_SIMPLEX,
#             0.6,
#             (255, 0, 0),
#             2
#         )
        
#         # Draw ROI center point on full image
#         cv2.circle(color_image, (x_roi_center, y_roi_center), 5, (0, 0, 255), -1)
#         cv2.putText(
#             color_image,                   
#             f"[ROI center point] Depth: {distance:.3f} m",
#             (20, 40),
#             cv2.FONT_HERSHEY_SIMPLEX,
#             0.6,
#             (0, 0, 255),
#             2
#         )

#         cam.imshow("color_image", color_image)
#         cam.imshow("depth_image", depth_colorMap)

#         key = d435i.waitKey(1)
#         if key == 27:
#             break

# finally:
#     cam.stopPipeline()
#     cam.destroyAllWindows()

<span style="
    color:gray;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:30px;
    letter-spacing:1px;
    line-height:1.2;
">
Test Depth Camera with ROI: Detect Colored objects
</span>

In [30]:
# # --- Select camera location ----------------------------
# location = "head"       # head / wrist_R / wrist_L
# # -------------------------------------------------------

# if location == "head":
#     cam = d435i
# elif location == "wrist_R":
#     cam = d405_R
# elif location == "wrist_L":
#     cam = d405_L
    
# # ============================================================
# # ROI setting 
# # Pixel Origin: (0,0) = 왼쪽 위 꼭짓점
# # Posture: PACK_POS
# # ============================================================
# # See Table Top
# pixel_roi_x = pixel_roi_x_tableTop
# pixel_roi_y = pixel_roi_y_tableTop

# cam.startPipeline()

# try:
#     while True:
#         color_frame, depth_frame = cam.getRawFrames()

#         if color_frame is None or depth_frame is None:
#             continue

#         color_image, depth_image = cam.frameToImage(color_frame, depth_frame)
#         depth_colorMap = cam.convertDepthToColorMap(depth_image, alpha=0.03)

#         detected_objects, color_image, mask = cam.detectColorObjectContour3D(
#             color_image=color_image,
#             depth_frame=depth_frame,
#             mask_list=ysw.HSV_MASK_RED,
#             minimum_area=object_minimum_area,     
#             maximum_area=object_maximum_area,       
#             maximum_depth=object_maximum_distance,  
#             pixel_roi_x=pixel_roi_x,
#             pixel_roi_y=pixel_roi_y,
#         )

#         for obj in detected_objects:
#             print(
#                 f"[object {obj['id']}] "
#                 f"center_3d={np.round(obj['center_3d'], 3)}, "
#                 f"bbox_3d_size={np.round(obj['bbox_3d_size'], 3)}, "
#                 f"area_2d={np.round(obj['area_2d'], 3)}"
#             )

#         cam.imshow("color_image", color_image)
#         cam.imshow("depth_image", depth_colorMap)
#         cam.imshow("mask", mask)

#         key = cam.waitKey(1)
#         if key == 27:
#             break

# finally:
#     cam.stopPipeline()
#     cam.destroyAllWindows()